# Feature Engineering for GNN-Based Portfolio Model

This notebook implements **Step 1: node feature preparation** starting from raw per-ticker CSV files stored in the `data/` folder.

Primary objectives:
- Load OHLCV data from individual ticker CSV files.
- Align all tickers on their overlapping date range.
- Build an aligned close-price panel.
- Compute basic return and volatility features.
- Optionally prepare rolling covariance matrices for later graph-based modeling steps.
- Provide a reusable helper function to build node features for a selected date.

## 1. Imports & configuration

In [1]:
from pathlib import Path
import warnings
import json

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

DATA_DIR = Path("../../data/")
OHLCV_DIR = DATA_DIR/"OHLCV"

VOLATILITY_WINDOWS = [5, 20]
COVARIANCE_WINDOW = 20

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

csv_files = sorted(OHLCV_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in {DATA_DIR.resolve()}")
print([p.name for p in csv_files[:10]])

Found 30 CSV files in C:\Users\mirae\Desktop\Personalization_Engine\data
['ABUK.csv', 'ADIB.csv', 'AMOC.csv', 'ARCC.csv', 'BTFH.csv', 'CCAP.csv', 'CIEB.csv', 'COMI.csv', 'EAST.csv', 'EGAL.csv']


## 2. Load and merge OHLCV data

In [2]:
def standardize_ohlcv_frame(file_path: Path) -> pd.DataFrame:
    ticker = file_path.stem.upper()
    df = pd.read_csv(file_path)

    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df.dropna(subset=["datetime", "close"]).copy()
    df = df.sort_values("datetime")
    df = df.drop_duplicates(subset=["datetime"], keep="last")
    df = df.set_index("datetime")

    numeric_cols = [col for col in ["open", "high", "low", "close", "volume"] if col in df.columns]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["ticker"] = ticker
    return df


raw_data = {}
load_summary = []

for file_path in csv_files:
    ticker = file_path.stem.upper()
    ticker_df = standardize_ohlcv_frame(file_path)
    raw_data[ticker] = ticker_df
    load_summary.append(
        {
            "ticker": ticker,
            "rows": len(ticker_df),
            "start": ticker_df.index.min(),
            "end": ticker_df.index.max(),
            "null_close": int(ticker_df["close"].isna().sum()),
        }
    )

summary_df = pd.DataFrame(load_summary).sort_values("ticker").reset_index(drop=True)

print("Per-ticker date coverage:")
display(summary_df[["ticker", "rows", "start", "end"]])

# Convert start/end to datetime if needed
summary_df["start"] = pd.to_datetime(summary_df["start"])
summary_df["end"]   = pd.to_datetime(summary_df["end"])

# Filter: keep only tickers whose earliest date is on/before 2018-01-01
cutoff_date = pd.Timestamp("2017-01-01")
tickers_to_keep = summary_df.loc[summary_df["start"] <= cutoff_date, "ticker"].tolist()
tickers_dropped = summary_df.loc[summary_df["start"] > cutoff_date, "ticker"].tolist()

print(f"\nTickers to keep (start <= {cutoff_date.date()}): {len(tickers_to_keep)}")
print(tickers_to_keep)
print(f"\nTickers dropped (start > {cutoff_date.date()}): {len(tickers_dropped)}")
print(tickers_dropped)

# Restrict raw_data to the filtered tickers
raw_data = {tic: raw_data[tic] for tic in tickers_to_keep}

Per-ticker date coverage:


,ticker,rows,start,end
0,ABUK,2735,2014-11-11 10:00:00,2026-03-11 10:00:00
1,ADIB,2735,2014-12-04 10:00:00,2026-03-11 10:00:00
2,AMOC,2735,2014-12-10 10:00:00,2026-03-11 10:00:00
3,ARCC,2735,2014-12-02 10:00:00,2026-03-11 10:00:00
4,BTFH,2735,2014-12-10 10:00:00,2026-03-11 10:00:00
5,CCAP,2735,2014-12-10 10:00:00,2026-03-11 10:00:00
6,CIEB,2735,2014-12-08 10:00:00,2026-03-11 10:00:00
7,COMI,2735,2014-12-10 10:00:00,2026-03-11 10:00:00
8,EAST,2735,2014-12-02 10:00:00,2026-03-11 10:00:00
9,EGAL,2735,2014-11-02 10:00:00,2026-03-11 10:00:00



Tickers to keep (start <= 2017-01-01): 26
['ABUK', 'ADIB', 'AMOC', 'ARCC', 'BTFH', 'CCAP', 'CIEB', 'COMI', 'EAST', 'EGAL', 'EMFD', 'ETEL', 'GBCO', 'HRHO', 'JUFO', 'MASR', 'MCQE', 'MFPC', 'ORAS', 'ORHD', 'ORWE', 'PHDC', 'RAYA', 'SKPC', 'TMGH', 'VLMR']

Tickers dropped (start > 2017-01-01): 4
['FWRY', 'ISPH', 'RMDA', 'VLMRA']


## 4. Drop late‑start tickers (start > 2017‑01‑01), then align the remaining tickers on their overlapping date range

In [3]:
close_series = []

for ticker, df in raw_data.items():
    series = df["close"].rename(ticker)
    close_series.append(series)

prices_df = pd.concat(close_series, axis=1, join="inner")
prices_df = prices_df.sort_index()
prices_df = prices_df.dropna(how="any")

first_year = int(prices_df.index.min().year)
last_year = int(prices_df.index.max().year)
panel_filename = f"EGX30_OHLCV({first_year}-{last_year}).csv"

prices_df.to_csv(DATA_DIR/panel_filename)

print(f"Aligned price panel shape: {prices_df.shape}")
print(f"Date range: {prices_df.index.min()} -> {prices_df.index.max()}")
prices_df.head()

Aligned price panel shape: (2297, 26)
Date range: 2016-09-07 11:00:00 -> 2026-03-11 10:00:00


,ABUK,ADIB,AMOC,ARCC,BTFH,CCAP,CIEB,COMI,EAST,EGAL,EMFD,ETEL,GBCO,HRHO,JUFO,MASR,MCQE,MFPC,ORAS,ORHD,ORWE,PHDC,RAYA,SKPC,TMGH,VLMR
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-09-07 11:00:00,7.593992,1.102851,2.567997,6.56,2.790671,0.98,5.919846,17.867440,3.634067,3.445453,2.33,9.86,2.26,5.542218,3.680,3.541629,29.869119,3.170843,69.260002,0.992,4.553995,2.431124,0.089176,5.638882,5.68,0.389439
2016-09-08 11:00:00,7.593992,1.142130,2.521331,6.58,2.790671,0.99,5.907409,18.001592,3.700306,3.424241,2.38,9.70,2.30,5.488885,3.656,3.551386,29.869119,2.884037,69.779999,1.002,4.661995,2.441007,0.090588,5.638882,5.65,0.389439
2016-09-14 11:00:00,7.593992,1.130044,2.560664,6.44,2.790671,0.98,5.907409,17.849312,3.700306,3.424241,2.35,9.64,2.27,5.466662,3.632,3.534312,29.869119,2.604422,68.459999,0.980,4.823995,2.431124,0.095529,5.597215,5.61,0.389439
2016-09-15 11:00:00,7.599992,1.111915,2.533331,6.28,2.790671,0.96,5.904922,17.443234,3.613914,3.424241,2.30,9.56,2.23,5.413329,3.456,3.487968,29.869119,2.574064,70.610001,0.960,4.703995,2.401477,0.097882,5.532400,5.41,0.380785
2016-09-18 11:00:00,7.662659,1.093786,2.505331,6.16,2.799293,0.94,5.907409,17.381597,3.613914,3.424241,2.29,9.31,2.23,5.439996,3.352,3.448942,28.323680,2.562879,70.010002,0.942,4.631995,2.361946,0.099765,5.467586,5.24,0.380785


## 4. Compute returns & volatility features

In [4]:
# Daily simple returns and log returns
returns_df = prices_df.pct_change()
log_returns_df = np.log(prices_df / prices_df.shift(1))

# Cross-sectional feature tables indexed by date, with ticker columns
volatility_features = {
    f"vol_{window}d": returns_df.rolling(window).std()
    for window in VOLATILITY_WINDOWS
}

# Additional optional per-ticker features that are often useful in node construction
momentum_features = {
    "ret_1d": log_returns_df,
    "ret_5d": log_returns_df.rolling(5).sum(),
    "ret_20d": log_returns_df.rolling(20).sum(),
}

feature_preview = pd.DataFrame({
    "latest_close": prices_df.iloc[-1],
    "latest_ret_1d": momentum_features["ret_1d"].iloc[-1],
    "latest_ret_5d": momentum_features["ret_5d"].iloc[-1],
    "latest_vol_20d": volatility_features["vol_20d"].iloc[-1],
}).sort_index()

historical_feature_panels = {
    "close": prices_df,
    "log_close": np.log(prices_df),
    "ret_1d": log_returns_df,
    "ret_5d": momentum_features["ret_5d"],
    "ret_20d": momentum_features["ret_20d"],
    "vol_5d": volatility_features["vol_5d"],
    "vol_20d": volatility_features["vol_20d"],
}

# --- 1. Enforce Consistent Ticker Ordering and Date Alignment ---
tickers = sorted(prices_df.columns)
common_index = prices_df.index

# Intersect all indices to find common dates
for name, panel in historical_feature_panels.items():
    common_index = common_index.intersection(panel.index)

# Reindex all panels to common index and sorted tickers
for name, panel in historical_feature_panels.items():
    historical_feature_panels[name] = panel.loc[common_index, tickers]

prices_df = historical_feature_panels["close"]

# --- 2. Compute and Store Feature Normalization Stats ---
FEATURE_NAMES = ["close", "log_close", "ret_1d", "ret_5d", "ret_20d", "vol_5d", "vol_20d"]
feature_means = {}
feature_stds = {}

for f in FEATURE_NAMES:
    vals = historical_feature_panels[f].values.flatten()
    vals = vals[~np.isnan(vals)]
    feature_means[f] = float(vals.mean())
    feature_stds[f] = float(vals.std() + 1e-8)  # avoid division by zero

stats_path = DATA_DIR / "feature_stats.json"
with open(stats_path, "w") as f:
    json.dump({"means": feature_means, "stds": feature_stds}, f, indent=2)
print(f"Saved feature normalization stats to {stats_path}")

ticker_feature_output_dir = DATA_DIR / "C+Features"
ticker_feature_output_dir.mkdir(parents=True, exist_ok=True)

ticker_feature_paths = {}

for ticker in prices_df.columns:
    ticker_features = pd.DataFrame(index=prices_df.index)
    ticker_features.index.name = "datetime"

    # Use specific FEATURE_NAMES to enforce column order
    for feature_name in FEATURE_NAMES:
        if feature_name in historical_feature_panels:
            ticker_features[feature_name] = historical_feature_panels[feature_name][ticker]

    ticker_features = ticker_features.dropna(how="any")
    output_path = ticker_feature_output_dir / f"{ticker}_Features.csv"
    ticker_features.to_csv(output_path, index=True)
    ticker_feature_paths[ticker] = output_path

print(f"Saved {len(ticker_feature_paths)} per-ticker historical feature files to: {ticker_feature_output_dir.resolve()}")

feature_preview.head(10)

Saved feature normalization stats to ..\..\data\feature_stats.json
Saved 26 per-ticker historical feature files to: C:\Users\mirae\Desktop\Personalization_Engine\data\C+Features


,latest_close,latest_ret_1d,latest_ret_5d,latest_vol_20d
ABUK,85.00,0.061875,0.077771,0.049449
ADIB,41.00,0.050780,0.080998,0.038299
AMOC,8.88,0.035534,0.164907,0.045242
ARCC,49.01,-0.004885,-0.020399,0.020072
BTFH,2.96,0.000000,0.006780,0.023233
CCAP,3.65,0.011019,0.076851,0.027658
CIEB,23.30,0.048812,0.079935,0.024055
COMI,125.30,-0.032970,0.030797,0.028083
EAST,35.00,-0.078283,-0.078283,0.026917
EGAL,295.00,0.062958,0.084746,0.048381


## 5. Compute covariance matrices for later steps

In [5]:
covariance_matrices = {}
valid_cov_dates = returns_df.index[COVARIANCE_WINDOW - 1 :]

for dt in valid_cov_dates:
    loc = returns_df.index.get_loc(dt)
    window_slice = returns_df.iloc[loc - COVARIANCE_WINDOW + 1 : loc + 1]
    if window_slice.isna().any().any():
        continue
    covariance_matrices[dt] = window_slice.cov()

print(f"Computed {len(covariance_matrices)} rolling covariance matrices using a {COVARIANCE_WINDOW}-day window.")

if covariance_matrices:
    sample_cov_date = next(iter(covariance_matrices))
    print(f"Sample covariance date: {sample_cov_date}")
    display(covariance_matrices[sample_cov_date].iloc[:5, :5])

Computed 2277 rolling covariance matrices using a 20-day window.
Sample covariance date: 2016-10-12 11:00:00


,ABUK,ADIB,AMOC,ARCC,BTFH
ABUK,0.000177,0.000079,-0.000040,1.310860e-04,1.463784e-06
ADIB,0.000079,0.000349,-0.000045,2.498421e-04,-2.243957e-06
AMOC,-0.000040,-0.000045,0.000506,1.422896e-04,-2.444536e-06
ARCC,0.000131,0.000250,0.000142,5.274182e-04,-7.442912e-07
BTFH,0.000001,-0.000002,-0.000002,-7.442912e-07,4.772807e-07


## 6. Helper function: `build_node_features(...)`

In [6]:
def build_node_features(
    as_of_date,
    prices: pd.DataFrame,
    returns: pd.DataFrame,
    volatility_dict: dict,
    momentum_dict: dict | None = None,
    include_price_level: bool = True,
    dropna: bool = True,
) -> pd.DataFrame:
    """
    Build a node-feature matrix for all tickers at a single date.

    Parameters
    ----------
    as_of_date : str or pd.Timestamp
        Date at which to extract node features.
    prices : pd.DataFrame
        Aligned close-price panel of shape [num_days, num_stocks].
    returns : pd.DataFrame
        Daily return panel aligned with prices.
    volatility_dict : dict[str, pd.DataFrame]
        Mapping from feature name to panel, e.g. {"vol_5d": df, "vol_20d": df}.
    momentum_dict : dict[str, pd.DataFrame] | None
        Optional mapping of return-based features such as ret_5d, ret_20d.
    include_price_level : bool
        If True, include close price and log-close in the feature matrix.
    dropna : bool
        If True, drop rows containing missing values.

    Returns
    -------
    pd.DataFrame
        Node features indexed by ticker, shape [num_stocks, num_features].
    """
    as_of_date = pd.Timestamp(as_of_date)

    if as_of_date not in prices.index:
        raise KeyError(f"Date {as_of_date} not found in aligned price panel.")

    features = pd.DataFrame(index=prices.columns)
    features.index.name = "ticker"

    if include_price_level:
        features["close"] = prices.loc[as_of_date]
        features["log_close"] = np.log(prices.loc[as_of_date])

    features["ret_1d"] = returns.loc[as_of_date]

    for feature_name, feature_panel in volatility_dict.items():
        features[feature_name] = feature_panel.loc[as_of_date]

    if momentum_dict is not None:
        for feature_name, feature_panel in momentum_dict.items():
            if feature_name == "ret_1d":
                continue
            features[feature_name] = feature_panel.loc[as_of_date]

    if dropna:
        features = features.dropna(how="any")

    return features.sort_index()


# Example usage on the latest date with enough history
candidate_dates = returns_df.dropna().index
if len(candidate_dates) == 0:
    candidate_dates = prices_df.index

example_feature_date = candidate_dates[-1]
node_features_example = build_node_features(
    as_of_date=example_feature_date,
    prices=prices_df,
    returns=returns_df,
    volatility_dict=volatility_features,
    momentum_dict=momentum_features,
)

print(f"Example node feature date: {example_feature_date}")
print(f"Node feature matrix shape: {node_features_example.shape}")
node_features_example.head()

Example node feature date: 2026-03-11 10:00:00
Node feature matrix shape: (26, 7)


,close,log_close,ret_1d,vol_5d,vol_20d,ret_5d,ret_20d
ticker,,,,,,,
ABUK,85.00,4.442651,0.063830,0.070287,0.049449,0.077771,0.200605
ADIB,41.00,3.713572,0.052091,0.043769,0.038299,0.080998,0.088431
AMOC,8.88,2.183802,0.036173,0.085467,0.045242,0.164907,0.222299
ARCC,49.01,3.892024,-0.004873,0.011076,0.020072,-0.020399,0.030874
BTFH,2.96,1.085189,0.000000,0.013181,0.023233,0.006780,-0.049433


## 7. Sanity checks

In [7]:
print("=== Sanity checks ===")
print(f"Number of tickers loaded: {len(raw_data)}")
print(f"Tickers in aligned panel: {len(prices_df.columns)}")
print(f"Aligned panel shape: {prices_df.shape}")
print(f"Returns shape: {returns_df.shape}")
print(f"First aligned date: {prices_df.index.min()}")
print(f"Last aligned date: {prices_df.index.max()}")
print(f"No missing values in prices_df: {not prices_df.isna().any().any()}")
print(f"Panel saved to: {DATA_DIR/panel_filename}")
print(f"Feature columns: {list(node_features_example.columns)}")

assert prices_df.shape[1] == len(raw_data), "Some tickers were lost during alignment."
assert prices_df.index.is_monotonic_increasing, "Price index is not sorted."
assert not prices_df.isna().any().any(), "Aligned price panel contains missing values."
print(f"Final number of tickers after dropping late-start stocks: {len(prices_df.columns)}")
prices_df.tail()

=== Sanity checks ===
Number of tickers loaded: 26
Tickers in aligned panel: 26
Aligned panel shape: (2297, 26)
Returns shape: (2297, 26)
First aligned date: 2016-09-07 11:00:00
Last aligned date: 2026-03-11 10:00:00
No missing values in prices_df: True
Panel saved to: ..\..\data\EGX30_OHLCV(2016-2026).csv
Feature columns: ['close', 'log_close', 'ret_1d', 'vol_5d', 'vol_20d', 'ret_5d', 'ret_20d']
Final number of tickers after dropping late-start stocks: 26


ticker,ABUK,ADIB,AMOC,ARCC,BTFH,CCAP,CIEB,COMI,EAST,EGAL,EMFD,ETEL,GBCO,HRHO,JUFO,MASR,MCQE,MFPC,ORAS,ORHD,ORWE,PHDC,RAYA,SKPC,TMGH,VLMR
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-03-05 10:00:00,77.93,38.65,7.61,49.77,2.96,3.58,21.69,126.00,38.35,267.50,9.25,88.00,27.00,26.10,26.67,5.32,188.99,39.65,467.57,24.61,22.33,9.24,5.34,17.30,80.59,0.700
2026-03-08 10:00:00,87.00,37.69,8.81,49.04,2.94,3.72,21.72,121.91,37.89,302.85,9.11,86.00,25.69,26.01,26.50,5.42,185.20,43.56,464.00,24.20,23.00,8.75,5.37,19.25,77.89,0.705
2026-03-09 10:00:00,84.10,36.55,9.31,48.55,2.90,3.72,21.38,123.00,37.59,285.00,8.92,82.13,24.64,25.78,26.33,5.50,182.35,43.30,458.00,25.02,23.00,8.67,5.49,18.70,76.50,0.705
2026-03-10 10:00:00,79.90,38.97,8.57,49.25,2.96,3.61,22.19,129.50,37.85,277.00,9.21,84.01,25.90,26.68,26.58,5.60,183.69,41.90,464.97,25.37,23.06,8.90,5.83,18.01,79.80,0.710
2026-03-11 10:00:00,85.00,41.00,8.88,49.01,2.96,3.65,23.30,125.30,35.00,295.00,9.22,84.80,25.99,26.58,26.51,5.68,186.82,43.51,469.70,25.43,22.87,8.80,5.73,18.35,79.95,0.710
